# ⚖️ SO SÁNH ĐỐI ĐẦU TOÀN DIỆN: ROI Refiner vs FireGrounder V3 vs YOLO26-Pose

Notebook này thực hiện đánh giá toàn diện, khách quan giữa 3 thế hệ mô hình giải quyết bài toán định vị gốc lửa:
1. **Mô hình A — Two-Stage ROI Refiner (`best_roi.pth`):** MobileNetV4 cắt vùng ảnh vuông (crop 70%) quanh điểm thô để tinh chỉnh tọa độ chân lửa.
2. **Mô hình B — One-Stage Dual-Head FireGrounder V3 (`best_v3_fpn.pth`):** MobileNetV4 + FPN phân loại có/không cháy và dự đoán Heatmap/Regression toàn khung ảnh 256x256.
3. **Mô hình C — One-Stage Single-Keypoint YOLO26-Pose (`best.pt`):** Kiến trúc Pose tiên tiến nhất, đồng thời phát hiện Bounding Box và khóa chặt Keypoint gốc lửa duy nhất trong 1 lần forward.

### Tiêu chí so sánh:
- **Độ chính xác định vị:** MAE (Mean Absolute Error - pixel), Median Error, PCK@5, PCK@10, PCK@25.
- **Tốc độ & Độ nhẹ:** Dung lượng Checkpoint (MB), Số tham số (Parameters), Độ trễ suy luận (ms/ảnh), FPS.
- **Chất lượng hiển thị trực quan:** Các điểm chốt (Check Point) được thể hiện **đậm nét** (không phóng to kích thước, viền đen tương phản cao, đổ bóng chữ rõ ràng).


In [ ]:
# Cell 1: KHỞI TẠO VÀ TẢI CẢ 2 MÔ HÌNH VÀO BỘ NHỚ
import sys, os, time
from pathlib import Path
import torch
import numpy as np
from PIL import Image
from ultralytics import YOLO

# 1. Chuẩn hóa thư mục gốc dự án
NOTEBOOK_DIR = Path.cwd()
ROOT_DIR = NOTEBOOK_DIR if (NOTEBOOK_DIR / 'models').exists() else NOTEBOOK_DIR.parent
OUTPUTS_DIR = ROOT_DIR / 'outputs'
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

# 2. Tự động thêm các thư mục module vào sys.path
for mod_p in [ROOT_DIR / 'models' / 'ROI_Refiner', ROOT_DIR / 'models' / 'FireGrounder_V3', ROOT_DIR / 'LAB_SAM']:
    if mod_p.exists() and str(mod_p) not in sys.path:
        sys.path.insert(0, str(mod_p))

from narrow_localizer import ROIRefinerInference

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print(f'🚀 Thiết bị tính toán: {device}')

# 3. Tải Model A: ROI Refiner (MobileNetV4)
roi_candidates = [
    ROOT_DIR / 'models' / 'ROI_Refiner' / 'best_roi.pth',
    ROOT_DIR / 'best_roi.pth',
    ROOT_DIR / 'LAB_SAM' / 'best_roi.pth'
]
roi_path = next((p for p in roi_candidates if p.exists()), None)
assert roi_path is not None, 'Không tìm thấy best_roi.pth!'
roi_refiner = ROIRefinerInference(str(roi_path), device=device)
roi_size_mb = roi_path.stat().st_size / 1e6
roi_params = sum(p.numel() for p in roi_refiner.model.parameters())
print(f'✅ Đã tải Model A (ROI Refiner): {roi_path.name} | Dung lượng: {roi_size_mb:.2f} MB | Params: {roi_params/1e6:.2f}M')

# 4. Tải Model B: YOLO26-Pose (One-Stage Single-Keypoint)
yolo_candidates = [
    ROOT_DIR / 'models' / 'YOLO26_Pose' / 'best.pt',
    ROOT_DIR / 'models' / 'YOLO26_Pose' / 'training_runs' / 'yolo26n_fire_base' / 'weights' / 'best.pt',
    ROOT_DIR / 'runs' / 'pose' / 'fire_pose_runs' / 'yolo26n_fire_base' / 'weights' / 'best.pt'
]
yolo_path = next((p for p in yolo_candidates if p.exists()), None)
assert yolo_path is not None, 'Không tìm thấy best.pt của YOLO26-Pose!'
yolo_model = YOLO(str(yolo_path))
yolo_size_mb = yolo_path.stat().st_size / 1e6
yolo_params = sum(p.numel() for p in yolo_model.model.parameters())
print(f'✅ Đã tải Model B (YOLO26-Pose): {yolo_path.name} | Dung lượng: {yolo_size_mb:.2f} MB | Params: {yolo_params/1e6:.2f}M')


In [ ]:
# Cell 2: ĐÁNH GIÁ ĐỐI ĐẦU TRỰC TIẾP TRÊN TẬP TEST NGUYÊN BẢN (TEST SET)
# Đo lường sai số khoảng cách Euclidean (Pixel Error) trên cùng một tập ảnh và Ground Truth thật
import sys, os, time, json
from pathlib import Path
import torch
import numpy as np
from PIL import Image
from ultralytics import YOLO

# Tự động import và tải model nếu người dùng chạy thẳng Cell 2
for mod_p in [ROOT_DIR / 'models' / 'ROI_Refiner', ROOT_DIR / 'models' / 'FireGrounder_V3', ROOT_DIR / 'LAB_SAM']:
    if mod_p.exists() and str(mod_p) not in sys.path:
        sys.path.insert(0, str(mod_p))
from narrow_localizer import ROIRefinerInference

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'

if 'roi_refiner' not in globals():
    r_path = next((p for p in [ROOT_DIR / 'models' / 'ROI_Refiner' / 'best_roi.pth', ROOT_DIR / 'best_roi.pth'] if p.exists()), None)
    roi_refiner = ROIRefinerInference(str(r_path), device=device)
    roi_size_mb = r_path.stat().st_size / 1e6
    roi_params = sum(p.numel() for p in roi_refiner.model.parameters())

if 'yolo_model' not in globals():
    y_path = next((p for p in [ROOT_DIR / 'models' / 'YOLO26_Pose' / 'best.pt', ROOT_DIR / 'models' / 'YOLO26_Pose' / 'training_runs' / 'yolo26n_fire_base' / 'weights' / 'best.pt'] if p.exists()), None)
    yolo_model = YOLO(str(y_path))
    yolo_size_mb = y_path.stat().st_size / 1e6
    yolo_params = sum(p.numel() for p in yolo_model.model.parameters())

# Tìm file dataset_labels.json
labels_candidates = [
    ROOT_DIR / 'datasets' / 'fire_ground_dataset' / 'dataset_labels.json',
    ROOT_DIR / 'fire_ground_dataset' / 'dataset_labels.json'
]
labels_path = next((p for p in labels_candidates if p.exists()), None)
assert labels_path is not None, 'Không tìm thấy dataset_labels.json!'

with open(labels_path, 'r', encoding='utf-8') as f:
    raw_labels = json.load(f)

test_items = [item for item in raw_labels if '/test/' in item['image_path'].replace('\\', '/') and item['has_fire'] == 1]
print(f'📊 Tìm thấy {len(test_items)} mẫu test có lửa trong tập dữ liệu chuẩn.')

# Chạy benchmark đo tốc độ và sai số
roi_errors, yolo_errors = [], []
roi_times, yolo_times = [], []
test_records = []

print('⏳ Đang tiến hành suy luận đối đầu song song (Inference Benchmark)...')
for idx, item in enumerate(test_items):
    stem = Path(item['image_path']).name
    img_candidates = [
        ROOT_DIR / 'datasets' / 'dataset_fire_pose' / 'images' / 'test' / stem,
        ROOT_DIR / 'dataset_fire_pose' / 'images' / 'test' / stem
    ]
    img_path = next((p for p in img_candidates if p.exists()), None)
    if img_path is None: continue
    
    pil_img = Image.open(img_path).convert('RGB')
    W, H = pil_img.size
    gt_x = item['p_fire'][0] * W
    gt_y = item['p_fire'][1] * H
    
    # ─── ĐO MODEL B: YOLO26-Pose (One-Stage) ─────────────────────────────
    t0 = time.perf_counter()
    res = yolo_model.predict(str(img_path), conf=0.15, imgsz=384, device=device, verbose=False)[0]
    t_yolo = (time.perf_counter() - t0) * 1000
    yolo_times.append(t_yolo)
    
    if len(res.boxes) > 0 and res.keypoints is not None and len(res.keypoints.xy) > 0:
        yolo_pt = res.keypoints.xy[0][0].cpu().numpy()
        y_err = np.sqrt((yolo_pt[0] - gt_x)**2 + (yolo_pt[1] - gt_y)**2)
        yolo_errors.append(y_err)
        box = res.boxes.xyxy[0].cpu().numpy()
        coarse_pt = ((box[0] + box[2])/2, (box[1] + box[3])/2)
    else:
        coarse_pt = (W/2, H/2)
        
    # ─── ĐO MODEL A: ROI Refiner (Two-Stage) ──────────────────────────────
    t0 = time.perf_counter()
    refined = roi_refiner.refine(pil_img, coarse_pt)
    t_roi = (time.perf_counter() - t0) * 1000
    roi_times.append(t_roi)
    r_pt = refined.point
    r_err = np.sqrt((r_pt[0] - gt_x)**2 + (r_pt[1] - gt_y)**2)
    roi_errors.append(r_err)

print(f'🎉 Đã hoàn tất đánh giá đối đầu trên {len(roi_errors)} ảnh!')


In [ ]:
# Cell 3: BẢNG SO SÁNH TỔNG HỢP CÁC CHỈ SỐ KỸ THUẬT QUAN TRỌNG
def compute_metrics(errors, times, model_name, size_mb, params_m):
    errs = np.array(errors)
    return {
        'Dung lượng Checkpoint': f'{size_mb:.2f} MB',
        'Số tham số (M Params)': f'{params_m:.2f}M',
        'MAE Sai số trung bình (Pixel)': f'{np.mean(errs):.2f} px',
        'Median Sai số trung vị (Pixel)': f'{np.median(errs):.2f} px',
        'P90 Sai số percentile 90 (Pixel)': f'{np.percentile(errs, 90):.2f} px',
        'PCK@5px (Tỷ lệ sai số <= 5px)': f'{(errs <= 5).mean()*100:.1f}%',
        'PCK@10px (Tỷ lệ sai số <= 10px)': f'{(errs <= 10).mean()*100:.1f}%',
        'PCK@25px (Tỷ lệ sai số <= 25px)': f'{(errs <= 25).mean()*100:.1f}%',
        'Độ trễ suy luận trung bình (Latency)': f'{np.mean(times):.2f} ms/ảnh',
        'Tốc độ khung hình (Throughput)': f'{1000/np.mean(times):.1f} FPS'
    }

metrics_roi = compute_metrics(roi_errors, roi_times, 'ROI Refiner', roi_size_mb, roi_params/1e6)
metrics_yolo = compute_metrics(yolo_errors, yolo_times, 'YOLO26-Pose', yolo_size_mb, yolo_params/1e6)

try:
    import pandas as pd
    df_summary = pd.DataFrame([metrics_roi, metrics_yolo], index=['Model A: ROI Refiner (MobileNetV4)', 'Model B: YOLO26-Pose (One-Stage)']).T
    display(df_summary)
except Exception:
    print('=' * 85)
    print(f"{'Chỉ số đánh giá':<40} | {'ROI Refiner':<18} | {'YOLO26-Pose':<18}")
    print('-' * 85)
    for k in metrics_roi.keys():
        print(f"{k:<40} | {metrics_roi[k]:<18} | {metrics_yolo[k]:<18}")
    print('=' * 85)


In [ ]:
# Cell 4: VẼ BIỂU ĐỒ SO SÁNH TRỰC DIỆN (BOXPLOT & ĐƯỜNG CONG TÍCH LŨY PCK)
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

# 1. Boxplot sai số định vị (Pixel)
axes[0].boxplot([roi_errors, yolo_errors], tick_labels=['ROI Refiner (MobileNetV4)', 'YOLO26-Pose'], patch_artist=True,
                boxprops=dict(facecolor='lightblue'), medianprops=dict(color='red', linewidth=2))
axes[0].set_title('Phân bố sai số định vị chân lửa (Pixel Error)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Sai số (Pixel - Càng thấp càng tốt)')
axes[0].grid(True, linestyle='--', alpha=0.6)

# 2. Cumulative Error Curve (Tỷ lệ dự đoán chính xác theo bán kính R)
radii = np.linspace(0, 100, 101)
roi_acc = [np.mean(np.array(roi_errors) <= r) * 100 for r in radii]
yolo_acc = [np.mean(np.array(yolo_errors) <= r) * 100 for r in radii]

axes[1].plot(radii, roi_acc, label=f'ROI Refiner (AUC={np.mean(roi_acc):.1f})', color='blue', linewidth=2.5)
axes[1].plot(radii, yolo_acc, label=f'YOLO26-Pose (AUC={np.mean(yolo_acc):.1f})', color='orange', linewidth=2.5)
axes[1].axvline(x=20, color='gray', linestyle=':', label='R=20px Tolerance')
axes[1].set_title('Độ chính xác tích lũy theo bán kính dung sai (PCK Curve)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Bán kính dung sai R (Pixel)')
axes[1].set_ylabel('Tỷ lệ dự đoán chính xác (%)')
axes[1].legend(loc='lower right')
axes[1].grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
save_path = OUTPUTS_DIR / 'comparison_roi_vs_yolo26.png'
plt.savefig(save_path, dpi=150)
print(f'✅ Đã lưu biểu đồ phân tích vào: {save_path}')
plt.show()


In [ ]:
# Cell 5: TRỰC QUAN HÓA SO SÁNH 8 ẢNH TEST THỰC TẾ
import random, cv2
random.seed(42)
sample_indices = random.sample(range(len(test_items)), min(8, len(test_items)))

fig, axes = plt.subplots(4, 2, figsize=(14, 20))
axes = axes.flatten()

for i, idx in enumerate(sample_indices):
    item = test_items[idx]
    stem = Path(item['image_path']).name
    img_path = next((p for p in [ROOT_DIR / 'datasets' / 'dataset_fire_pose' / 'images' / 'test' / stem, ROOT_DIR / 'dataset_fire_pose' / 'images' / 'test' / stem] if p.exists()), None)
    if img_path is None: continue
    
    img_bgr = cv2.imread(str(img_path))
    H_img, W_img = img_bgr.shape[:2]
    gt_x = int(item['p_fire'][0] * W_img)
    gt_y = int(item['p_fire'][1] * H_img)
    
    pil_tmp = Image.fromarray(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    res = yolo_model.predict(img_bgr, conf=0.15, imgsz=384, device=device, verbose=False)[0]
    if len(res.boxes) > 0 and res.keypoints is not None and len(res.keypoints.xy) > 0:
        yolo_pt = res.keypoints.xy[0][0].cpu().numpy()
        box = res.boxes.xyxy[0].cpu().numpy()
        coarse_pt = ((box[0]+box[2])/2, (box[1]+box[3])/2)
    else:
        yolo_pt = (W_img/2, H_img/2)
        coarse_pt = (W_img/2, H_img/2)
        
    ref = roi_refiner.refine(pil_tmp, coarse_pt)
    roi_pt = ref.point
    
    ax = axes[i]
    ax.imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    ax.plot(gt_x, gt_y, 'go', markersize=12, label='Ground Truth (Chuẩn)', markeredgecolor='black', markeredgewidth=2)
    ax.plot(roi_pt[0], roi_pt[1], 'b^', markersize=11, label='ROI Refiner', markeredgecolor='black', markeredgewidth=1.5)
    ax.plot(yolo_pt[0], yolo_pt[1], 'rx', markersize=13, markeredgewidth=3, label='YOLO26-Pose')
    ax.set_title(f'#{i+1} {stem}', fontsize=11, fontweight='bold')
    ax.legend(loc='upper right', fontsize=9, framealpha=0.85)
    ax.axis('off')

plt.tight_layout()
save_path = OUTPUTS_DIR / 'visual_comparison_roi_vs_yolo26.png'
plt.savefig(save_path, dpi=150)
print(f'✅ Đã lưu ảnh so sánh trực quan vào: {save_path}')
plt.show()


In [ ]:
# Cell 6: SO SÁNH ĐỐI ĐẦU TOÀN DIỆN CẢ 3 MÔ HÌNH: ROI REFINER vs FIREGROUNDER V3 vs YOLO26-POSE
# (Tự động tải 3 mô hình, đo lường định lượng và trực quan hóa với Check Point đậm nét)
import sys, os, time, json, cv2
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torchvision.transforms.functional as TF
from ultralytics import YOLO

# 1. Thêm module vào sys.path
for p in [ROOT_DIR / 'models' / 'ROI_Refiner', ROOT_DIR / 'models' / 'FireGrounder_V3', ROOT_DIR / 'LAB_SAM']:
    if p.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

from firegrounder_v3 import FireGrounderV3, MEAN, STD
from narrow_localizer import ROIRefinerInference

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'

# 2. Tải cả 3 mô hình vào bộ nhớ
if 'roi_refiner' not in globals():
    r_path = next((p for p in [ROOT_DIR / 'models' / 'ROI_Refiner' / 'best_roi.pth', ROOT_DIR / 'best_roi.pth'] if p.exists()), None)
    roi_refiner = ROIRefinerInference(str(r_path), device=device)
    roi_size_mb = r_path.stat().st_size / 1e6
    roi_params = sum(p.numel() for p in roi_refiner.model.parameters())

if 'm_v3' not in globals():
    v3_path = next((p for p in [ROOT_DIR / 'models' / 'FireGrounder_V3' / 'best_v3_fpn.pth', ROOT_DIR / 'v3_outputs' / 'best_v3_fpn.pth'] if p.exists()), None)
    m_v3 = FireGrounderV3(pretrained=False).to(device)
    ckpt_v3 = torch.load(str(v3_path), map_location=device, weights_only=False)
    m_v3.load_state_dict(ckpt_v3.get('model_state_dict', ckpt_v3))
    m_v3.eval()
    v3_size_mb = v3_path.stat().st_size / 1e6
    v3_params = sum(p.numel() for p in m_v3.parameters())

if 'yolo_model' not in globals():
    y_path = next((p for p in [ROOT_DIR / 'models' / 'YOLO26_Pose' / 'best.pt', ROOT_DIR / 'models' / 'YOLO26_Pose' / 'training_runs' / 'yolo26n_fire_base' / 'weights' / 'best.pt'] if p.exists()), None)
    yolo_model = YOLO(str(y_path))
    yolo_size_mb = y_path.stat().st_size / 1e6
    yolo_params = sum(p.numel() for p in yolo_model.model.parameters())

# 3. Hàm vẽ điểm chốt đậm nét (Bold Check Point)
def draw_bold_checkpoint(img_rgb, pt, color, label, pos='top_right'):
    x, y = int(pt[0]), int(pt[1])
    h, w = img_rgb.shape[:2]
    x = max(15, min(w - 15, x))
    y = max(15, min(h - 15, y))
    cv2.circle(img_rgb, (x, y), 6, (0, 0, 0), 2)
    cv2.circle(img_rgb, (x, y), 5, color, -1)
    cv2.circle(img_rgb, (x, y), 1, (255, 255, 255), -1)
    offsets = {'top_right': (8, -6), 'bottom_right': (8, 16), 'top_left': (-48, -6), 'bottom_left': (-42, 16)}
    dx, dy = offsets.get(pos, (8, 4))
    lx, ly = max(5, min(w - 65, x + dx)), max(15, min(h - 10, y + dy))
    cv2.putText(img_rgb, label, (lx, ly), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 0, 0), 4, cv2.LINE_AA)
    cv2.putText(img_rgb, label, (lx, ly), cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2, cv2.LINE_AA)

# 4. Đánh giá định lượng trên 50 ảnh Test Set
labels_path = next((p for p in [ROOT_DIR / 'datasets' / 'fire_ground_dataset' / 'dataset_labels.json', ROOT_DIR / 'fire_ground_dataset' / 'dataset_labels.json'] if p.exists()), None)
with open(labels_path, 'r', encoding='utf-8') as f:
    all_records = json.load(f)
test_records = [r for r in all_records if '/test/' in r['image_path'].replace('\\', '/') and r['has_fire'] == 1]

roi_errs, v3_errs, yolo_errs = [], [], []
roi_times, v3_times, yolo_times = [], [], []

print('⏳ Đang đánh giá đối đầu 3 mô hình trên 50 ảnh test...')
for idx in range(min(50, len(test_records))):
    rec = test_records[idx]
    stem = Path(rec['image_path']).name
    img_path = next((p for p in [ROOT_DIR / 'datasets' / 'dataset_fire_pose' / 'images' / 'test' / stem, ROOT_DIR / 'dataset_fire_pose' / 'images' / 'test' / stem] if p.exists()), None)
    if img_path is None: continue
    
    pil_img = Image.open(img_path).convert('RGB')
    W_img, H_img = pil_img.size
    gt_x, gt_y = rec['p_fire'][0] * W_img, rec['p_fire'][1] * H_img
    
    # 1. YOLO26-Pose
    t0 = time.perf_counter()
    res = yolo_model.predict(str(img_path), conf=0.15, imgsz=384, device=device, verbose=False)[0]
    yolo_times.append((time.perf_counter() - t0) * 1000)
    if len(res.boxes) > 0 and res.keypoints is not None and len(res.keypoints.xy) > 0:
        yolo_pt = res.keypoints.xy[0][0].cpu().numpy()
        yolo_errs.append(np.sqrt((yolo_pt[0] - gt_x)**2 + (yolo_pt[1] - gt_y)**2))
        box = res.boxes.xyxy[0].cpu().numpy()
        coarse_pt = ((box[0]+box[2])/2, (box[1]+box[3])/2)
    else:
        yolo_errs.append(np.sqrt(W_img**2 + H_img**2)/2)
        coarse_pt = (W_img/2, H_img/2)
        
    # 2. ROI Refiner
    t0 = time.perf_counter()
    refined = roi_refiner.refine(pil_img, coarse_pt)
    roi_times.append((time.perf_counter() - t0) * 1000)
    roi_errs.append(np.sqrt((refined.point[0] - gt_x)**2 + (refined.point[1] - gt_y)**2))
    
    # 3. FireGrounder V3
    t0 = time.perf_counter()
    t_v3 = TF.normalize(TF.to_tensor(TF.resize(pil_img, [256, 256])), MEAN, STD).unsqueeze(0).to(device)
    with torch.no_grad():
        out_v3 = m_v3(t_v3)
        if isinstance(out_v3, dict): out_v3 = out_v3['pred']
        pred_v3 = out_v3[0].cpu().numpy()
    v3_times.append((time.perf_counter() - t0) * 1000)
    v3_errs.append(np.sqrt((pred_v3[1]*W_img - gt_x)**2 + (pred_v3[2]*H_img - gt_y)**2))

# 5. In bảng số liệu đối đầu 3 mô hình
def get_metrics_dict(errs, times_list, size_mb, params_m):
    arr = np.array(errs)
    return {
        'Dung lượng Model': f'{size_mb:.2f} MB',
        'Số tham số (M Params)': f'{params_m:.2f}M',
        'MAE Sai số TB (Pixel)': f'{np.mean(arr):.2f} px',
        'Median Sai số trung vị': f'{np.median(arr):.2f} px',
        'PCK@10px (Sai số <= 10px)': f'{(arr <= 10).mean()*100:.1f}%',
        'PCK@25px (Sai số <= 25px)': f'{(arr <= 25).mean()*100:.1f}%',
        'Độ trễ xử lý (Latency)': f'{np.mean(times_list):.2f} ms',
        'Tốc độ khung hình (FPS)': f'{1000/np.mean(times_list):.1f} FPS'
    }

metrics_roi = get_metrics_dict(roi_errs, roi_times, roi_size_mb, roi_params/1e6)
metrics_v3 = get_metrics_dict(v3_errs, v3_times, v3_size_mb, v3_params/1e6)
metrics_yolo = get_metrics_dict(yolo_errs, yolo_times, yolo_size_mb, yolo_params/1e6)

try:
    import pandas as pd
    df_3 = pd.DataFrame([metrics_roi, metrics_v3, metrics_yolo], index=['Model 1: ROI Refiner (MobileNetV4)', 'Model 2: FireGrounder V3 (FPN)', 'Model 3: YOLO26-Pose (One-Stage)']).T
    display(df_3)
except Exception:
    print('=' * 95)
    print(f"{'Chỉ số':<28} | {'1. ROI Refiner':<20} | {'2. FireGrounder V3':<20} | {'3. YOLO26-Pose':<20}")
    print('-' * 95)
    for k in metrics_roi.keys():
        print(f"{k:<28} | {metrics_roi[k]:<20} | {metrics_v3[k]:<20} | {metrics_yolo[k]:<20}")
    print('=' * 95)

# 6. Trực quan hóa 6 ảnh thực tế
num_vis = 6
fig, axes = plt.subplots(3, 2, figsize=(16, 18))
axes = axes.flatten()

vis_idx = 0
for rec in test_records:
    if vis_idx >= num_vis: break
    stem = Path(rec['image_path']).name
    img_path = next((p for p in [ROOT_DIR / 'datasets' / 'dataset_fire_pose' / 'images' / 'test' / stem, ROOT_DIR / 'dataset_fire_pose' / 'images' / 'test' / stem] if p.exists()), None)
    if img_path is None: continue
    
    img_bgr = cv2.imread(str(img_path))
    H_img, W_img = img_bgr.shape[:2]
    gt_x, gt_y = int(rec['p_fire'][0] * W_img), int(rec['p_fire'][1] * H_img)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    
    draw_bold_checkpoint(img_rgb, (gt_x, gt_y), (0, 255, 0), 'GT', pos='top_left')
    
    res = yolo_model.predict(img_bgr, conf=0.15, imgsz=384, device=device, verbose=False)[0]
    if len(res.boxes) > 0:
        box = res.boxes.xyxy[0].cpu().numpy()
        cv2.rectangle(img_rgb, (int(box[0]), int(box[1])), (int(box[2]), int(box[3])), (255, 140, 0), 2)
        if res.keypoints is not None and len(res.keypoints.xy) > 0:
            yk = res.keypoints.xy[0][0].cpu().numpy()
            draw_bold_checkpoint(img_rgb, (yk[0], yk[1]), (255, 30, 30), 'YOLO26', pos='bottom_right')
        coarse_pt = ((box[0]+box[2])/2, (box[1]+box[3])/2)
    else:
        coarse_pt = (W_img/2, H_img/2)
        
    pil_tmp = Image.fromarray(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    ref = roi_refiner.refine(pil_tmp, coarse_pt)
    draw_bold_checkpoint(img_rgb, (ref.point[0], ref.point[1]), (30, 144, 255), 'ROI', pos='top_right')
    
    t_v3 = TF.normalize(TF.to_tensor(TF.resize(pil_tmp, [256, 256])), MEAN, STD).unsqueeze(0).to(device)
    with torch.no_grad():
        out_v3 = m_v3(t_v3)
        if isinstance(out_v3, dict): out_v3 = out_v3['pred']
        pred_v3 = out_v3[0].cpu().numpy()
    v3_px, v3_py = int(pred_v3[1] * W_img), int(pred_v3[2] * H_img)
    draw_bold_checkpoint(img_rgb, (v3_px, v3_py), (255, 180, 0), 'V3', pos='bottom_left')
    
    axes[vis_idx].imshow(img_rgb)
    axes[vis_idx].set_title(f'{stem}\n[Xanh lá: GT | Xanh dương: ROI | Cam: V3 | Đỏ: YOLO26]', fontsize=11, fontweight='bold')
    axes[vis_idx].axis('off')
    vis_idx += 1

plt.tight_layout()
save_path = OUTPUTS_DIR / 'visual_comparison_3_models.png'
plt.savefig(save_path, dpi=150)
plt.show()
print(f'✅ Hoàn tất so sánh đối đầu 3 mô hình! Ảnh trực quan đã được lưu vào: {save_path}')


In [ ]:
# Cell 7: THỬ NGHIỆM THỰC CHIẾN ĐỐI ĐẦU 3 MÔ HÌNH TRÊN 30 ẢNH NGOẠI CẢNH (FIRE-SAMPLES)
# Thử thách Out-Of-Distribution: Khói đặc, đường phố, ban đêm, đèn đường lóa, người đi bộ
# Hiển thị 30 ảnh ngẫu nhiên, mỗi hàng 2 ảnh khổ lớn (ncols=2), Điểm chốt của cả 3 mô hình vẽ ĐẬM NÉT
import sys, os, time, random, cv2
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torchvision.transforms.functional as TF
from ultralytics import YOLO

# 1. Thêm module vào sys.path
for p in [ROOT_DIR / 'models' / 'ROI_Refiner', ROOT_DIR / 'models' / 'FireGrounder_V3', ROOT_DIR / 'LAB_SAM']:
    if p.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

from firegrounder_v3 import FireGrounderV3, MEAN, STD
from narrow_localizer import ROIRefinerInference

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'

# 2. Tìm thư mục ảnh fire-samples
sample_dir = next((p for p in [ROOT_DIR / 'datasets' / 'fire-samples', ROOT_DIR / 'fire-samples'] if p.exists()), None)
assert sample_dir is not None, 'Không tìm thấy thư mục fire-samples!'

exts = ['.jpg', '.jpeg', '.png', '.webp', '.bmp']
all_sample_imgs = [p for p in sample_dir.iterdir() if p.suffix.lower() in exts]
print(f'📂 Tổng số ảnh ngoại cảnh tìm thấy trong fire-samples: {len(all_sample_imgs)}')

# 3. Lấy ngẫu nhiên 30 ảnh
NUM_SAMPLES = min(30, len(all_sample_imgs))
random.seed(42)
selected_imgs = random.sample(all_sample_imgs, NUM_SAMPLES)

# 4. Hàm vẽ điểm chốt đậm nét
def draw_bold_checkpoint_fs(img_rgb, pt, color, label, pos='top_right'):
    x, y = int(pt[0]), int(pt[1])
    h, w = img_rgb.shape[:2]
    x = max(15, min(w - 15, x))
    y = max(15, min(h - 15, y))
    cv2.circle(img_rgb, (x, y), 6, (0, 0, 0), 2)
    cv2.circle(img_rgb, (x, y), 5, color, -1)
    cv2.circle(img_rgb, (x, y), 1, (255, 255, 255), -1)
    offsets = {'top_right': (8, -6), 'bottom_right': (8, 16), 'top_left': (-48, -6), 'bottom_left': (-42, 16)}
    dx, dy = offsets.get(pos, (8, 4))
    lx, ly = max(5, min(w - 65, x + dx)), max(15, min(h - 10, y + dy))
    cv2.putText(img_rgb, label, (lx, ly), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 0, 0), 4, cv2.LINE_AA)
    cv2.putText(img_rgb, label, (lx, ly), cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2, cv2.LINE_AA)

# 5. Vẽ lưới 30 ảnh: mỗi hàng 2 ảnh
cols = 2
rows = (NUM_SAMPLES + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(18, rows * 6.5))
axes = np.array(axes).flatten()

print(f'⏳ Đang phân tích và vẽ điểm chốt đậm nét cho {NUM_SAMPLES} ảnh ngoại cảnh...')
for i, img_path in enumerate(selected_imgs):
    img_bgr = cv2.imread(str(img_path))
    if img_bgr is None: continue
    H_img, W_img = img_bgr.shape[:2]
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    pil_tmp = Image.fromarray(img_rgb)
    
    # A. YOLO26-Pose
    res = yolo_model.predict(img_bgr, conf=0.15, imgsz=384, device=device, verbose=False)[0]
    if len(res.boxes) > 0:
        box = res.boxes.xyxy[0].cpu().numpy()
        cv2.rectangle(img_rgb, (int(box[0]), int(box[1])), (int(box[2]), int(box[3])), (255, 140, 0), 2)
        if res.keypoints is not None and len(res.keypoints.xy) > 0:
            yk = res.keypoints.xy[0][0].cpu().numpy()
            draw_bold_checkpoint_fs(img_rgb, (yk[0], yk[1]), (255, 30, 30), 'YOLO26', pos='bottom_right')
        coarse_pt = ((box[0]+box[2])/2, (box[1]+box[3])/2)
    else:
        coarse_pt = (W_img/2, H_img/2)
        
    # B. ROI Refiner
    ref = roi_refiner.refine(pil_tmp, coarse_pt)
    draw_bold_checkpoint_fs(img_rgb, (ref.point[0], ref.point[1]), (30, 144, 255), 'ROI', pos='top_right')
    
    # C. FireGrounder V3
    t_v3 = TF.normalize(TF.to_tensor(TF.resize(pil_tmp, [256, 256])), MEAN, STD).unsqueeze(0).to(device)
    with torch.no_grad():
        out_v3 = m_v3(t_v3)
        if isinstance(out_v3, dict): out_v3 = out_v3['pred']
        pred_v3 = out_v3[0].cpu().numpy()
    v3_px, v3_py = int(pred_v3[1] * W_img), int(pred_v3[2] * H_img)
    draw_bold_checkpoint_fs(img_rgb, (v3_px, v3_py), (255, 180, 0), 'V3', pos='bottom_left')
    
    axes[i].imshow(img_rgb)
    axes[i].set_title(f'#{i+1:02d} {img_path.name[:25]}\n[Xanh dương: ROI Refiner | Cam: V3 | Đỏ: YOLO26-Pose]', fontsize=10.5, fontweight='bold')
    axes[i].axis('off')

for j in range(NUM_SAMPLES, len(axes)):
    axes[j].axis('off')

plt.tight_layout()
save_path = OUTPUTS_DIR / 'fire_samples_3_models_comparison.png'
plt.savefig(save_path, dpi=120)
plt.show()
print(f'🎉 Hoàn tất! Ảnh kết quả 30 mẫu đã được lưu vào: {save_path}')
